In [ ]:
import torch
import torch.nn as nn

### Encoding

In [ ]:
EMBEDDING_DIM = 768
HIDDEN_DIM = 256
CONTEXT_LEN = 64

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
text = "hello, world"

token_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
token_ids

In [ ]:
tokenizer.decode(token_ids)

In [ ]:
from torch.utils.data import Dataset

class GPTDataset(Dataset):
    def __init__(self, text, tokenizer, max_len, stride):
        inputs_ids = []
        target_ids = []
        tokens_ids = tokenizer.encode(text)

        for i in range(0, len(tokens_ids) - max_len, stride):
            inputs_ids.append(tokens_ids[i:i+max_len])
            target_ids.append(tokens_ids[i+1:i+max_len+1])

        self.inputs_ids = torch.tensor(inputs_ids)
        self.target_ids = torch.tensor(target_ids)

    def __len__(self):
        return len(self.inputs_ids)

    def __getitem__(self, index):
        return self.inputs_ids[index], self.target_ids[index]

In [ ]:
from torch.utils.data import DataLoader

def create_dataloader(text, batch_size: int = 4, max_len = 256, stride = 128, shuffle = True, drop_last = True, num_workers = 0):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset(text, tokenizer, max_len, stride)
    dataloader = DataLoader(
        dataset,
        batch_size,
        shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

In [ ]:
with open('the-verdict.txt', mode='r', encoding='utf-8') as f:
    text = f.read()

dataloader = create_dataloader(text, batch_size=1, max_len=CONTEXT_LEN, stride=1, shuffle=False)
dataloader

In [ ]:
data_iter = iter(dataloader)
data_batch = next(data_iter)

In [ ]:
inputs_ids, target_ids = data_batch
inputs_ids

In [ ]:
token_encoding_layer = nn.Embedding(tokenizer.n_vocab, EMBEDDING_DIM)
token_encoding_layer

In [ ]:
token_embeddings = token_encoding_layer(inputs_ids)
token_embeddings

In [ ]:
pos_encoding_layer = nn.Embedding(CONTEXT_LEN, EMBEDDING_DIM)
pos_encoding_layer

In [ ]:
pos_embeddings = pos_encoding_layer(torch.arange(0, CONTEXT_LEN))
pos_embeddings

In [ ]:
inputs_embeddings = token_embeddings + pos_embeddings
inputs_embeddings

### Attention

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.q_W = nn.Linear(d_in, d_out, bias=True)
        self.k_W = nn.Linear(d_in, d_out, bias=True)
        self.v_W = nn.Linear(d_in, d_out, bias=True)

    def forward(self, x):
        q = self.q_W(x)
        k = self.k_W(x)
        v = self.v_W(x)

        attn_scores = q @ k.transpose(1, 2)
        attn_weights = torch.softmax(attn_scores / (q.shape[-1]**0.5), dim=-1)

        context_vecs = attn_weights @ v
        return context_vecs

self_attention = SelfAttention(EMBEDDING_DIM, HIDDEN_DIM)
self_attention

In [ ]:
self_attention(inputs_embeddings)

In [ ]:
class CasualAttention(nn.Module):
    def __init__(self, d_in: int, d_out: int, context_len: int, dropout_p: float = 0.5):
        super().__init__()
        self.q_W = nn.Linear(d_in, d_out, bias=True)
        self.k_W = nn.Linear(d_in, d_out, bias=True)
        self.v_W = nn.Linear(d_in, d_out, bias=True)
        self.dropout = nn.Dropout(dropout_p, inplace=False)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_len, context_len), diagonal=1),
            persistent=True
        )

    def forward(self, x):
        b, n_tokens, n_dim = x.shape

        q = self.q_W(x)
        k = self.k_W(x)
        v = self.v_W(x)

        attn_scores = q @ k.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:n_tokens, :n_tokens],
            -torch.inf
        )
        attn_weights = self.dropout(torch.softmax(attn_scores / (q.shape[-1]**0.5), dim=-1))

        context_vecs = attn_weights @ v
        return context_vecs

casual_attention = CasualAttention(EMBEDDING_DIM, HIDDEN_DIM, CONTEXT_LEN)
casual_attention

In [ ]:
casual_attention(inputs_embeddings)

### Training

In [ ]:
from tqdm import tqdm

class LLM(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int, hidden_dim: int, context_len: int):
        super().__init__()
        self.pos_encoding_layer = nn.Embedding(context_len, embedding_dim)
        self.token_encoding_layer = nn.Embedding(vocab_size, embedding_dim)
        self.casual_attention = CasualAttention(embedding_dim, hidden_dim, context_len, dropout_p=0.2)
        self.token_projection = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        b, n_tokens = x.shape

        inputs_embeddings = self.pos_encoding_layer(torch.arange(0, n_tokens)) + self.token_encoding_layer(x)
        output_embeddings = self.casual_attention(inputs_embeddings)

        return self.token_projection(output_embeddings)

with open('the-verdict.txt', mode='r', encoding='utf-8') as f:
    text = f.read()

llm = LLM(tokenizer.n_vocab, EMBEDDING_DIM, HIDDEN_DIM, CONTEXT_LEN)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(llm.parameters(), lr=1e-3)

for epoch in range(1, 100 + 1):
    losses = []
    for inputs_ids, target_ids in tqdm(create_dataloader(text, batch_size=1, max_len=4, stride=1, shuffle=False), desc="Processing training examples", unit="batch"):
        optimizer.zero_grad()
        logits = llm(inputs_ids)
        loss = loss_fn(logits.flatten(0, 1), target_ids.flatten())
        loss.backward()
        loss_curr = loss.item()
        losses.append(loss_curr)
        optimizer.step()
    if len(losses) > 5:
        loss_last = losses[-1]
        if abs(loss_last - loss_curr) < 1e-3:
            print(f"Early stopping, loss diff is less than 1e-3 between last epochs")
            break
    print(f"Epoch {epoch}: {loss.item()}")

### Inference

In [ ]:
with torch.no_grad():
    for inputs_ids, target_ids in create_dataloader("Hello, my lovely Daria! It's me, Dan.", batch_size=1,max_len=4, stride=1, shuffle=False):
        logits = llm(inputs_ids)
        probs = torch.softmax(logits, dim=-1)
        indices = torch.multinomial(probs.squeeze(0), num_samples=1, replacement=True)
        
        token_pred = tokenizer.decode(indices.squeeze(-1).tolist())
        token_true = tokenizer.decode(target_ids.squeeze(0).tolist())

        print(f"pred: {token_pred!r:<15} actual: {token_true!r}")